# Consolidated evaluation

Single source of truth for all model metrics. Loads prediction files from `outputs/`,
computes pointwise and ranking metrics, generates four visualisations.
Use numbers from this notebook for all presentation slides.
Inputs: `outputs/model*_predictions.csv`. Outputs: `outputs/viz_*.png`.

In [1]:
# Load prediction files, rescale Model 0 predictions, merge on job+candidate prefix
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score
from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

features = pd.read_csv('../outputs/features.csv')
m0 = pd.read_csv('../outputs/model0_predictions.csv')
ma = pd.read_csv('../outputs/model_a_predictions.csv')
mb = pd.read_csv('../outputs/model_b_predictions.csv')

for df_ in [features, m0, ma, mb]:
    df_['_pfx'] = df_['candidate_doc'].str[:120]
features['_orig_idx'] = features.index

# Rescale model0 predictions to matched_score range
y0_raw  = m0['model0_pred'].values
y0_true = m0['matched_score'].values
smin, smax = y0_true.min(), y0_true.max()
m0['model0_pred_rescaled'] = (
    (y0_raw - y0_raw.min()) / (y0_raw.max() - y0_raw.min()) * (smax - smin) + smin
)

df = (ma
      .merge(mb[['job_position_name','_pfx','model_b_pred']],
             on=['job_position_name','_pfx'], how='inner')
      .merge(m0[['job_position_name','_pfx','model0_pred_rescaled']],
             on=['job_position_name','_pfx'], how='inner')
      .merge(features[['job_position_name','_pfx']],
             on=['job_position_name','_pfx'], how='inner'))

df['job_id']  = df['job_position_name'].factorize()[0]
df['cand_id'] = df['_pfx'].factorize()[0]

print(f'Merged: {len(df)} rows')
print(f'Jobs: {df["job_id"].nunique()}   Candidates: {df["cand_id"].nunique()}')
print(f'Prediction columns: {[c for c in df.columns if "pred" in c]}')

Merged: 2027 rows
Jobs: 6   Candidates: 340
Prediction columns: ['model_a_pred', 'model_b_pred', 'model0_pred_rescaled']


## Pointwise evaluation

MAE and RMSE measure how close predicted scores are to true scores.
Spearman measures rank correlation — does the model rank candidates in the right order regardless of absolute score.

In [2]:
# Compute MAE, RMSE, Spearman for all three models on merged test set
y_true = df['matched_score'].values

MODELS = [
    ('Model 0', 'heuristic', 'model0_pred_rescaled'),
    ('Model A', 'TF-IDF',    'model_a_pred'),
    ('Model B', 'JobBERT',   'model_b_pred'),
]

metrics = {}
for name, label, col in MODELS:
    yp = df[col].values
    metrics[name] = {
        'label'     : label,
        'MAE'       : mean_absolute_error(y_true, yp),
        'RMSE'      : np.sqrt(mean_squared_error(y_true, yp)),
        'Spearman r': spearmanr(yp, y_true)[0],
    }

W = 16
headers = [f'{n} ({metrics[n]["label"]})' for n in metrics]
print(f'{"":16s}  ' + '  '.join(f'{h:>{W}}' for h in headers))
print('─' * (16 + len(metrics)*(W+2)))
for metric in ['MAE', 'RMSE', 'Spearman r']:
    row = '  '.join(f'{metrics[n][metric]:{W}.4f}' for n in metrics)
    print(f'{metric:16s}  {row}')

                  Model 0 (heuristic)  Model A (TF-IDF)  Model B (JobBERT)
──────────────────────────────────────────────────────────────────────
MAE                         0.4739            0.1274            0.1252
RMSE                        0.5136            0.1494            0.1477
Spearman r                  0.1653            0.4616            0.4535


## Ranking evaluation

NDCG@5 measures whether the best candidates appear in the top 5 for each job.
Computed per job then averaged. Also computed in the reverse direction:
for each candidate, rank jobs by predicted score and compute NDCG@5.

In [3]:
# NDCG@5 by job and by candidate; per-job breakdown for all models
def mean_ndcg(data, group_col, pred_col, k=5):
    scores = []
    for _, g in data.groupby(group_col):
        if len(g) < 2: continue
        true = g['matched_score'].values.reshape(1,-1)
        pred = g[pred_col].values.reshape(1,-1)
        scores.append(ndcg_score(true, pred, k=k))
    return float(np.mean(scores)), scores

ndcg_job  = {}
ndcg_cand = {}
for name, _, col in MODELS:
    ndcg_job[name],  _ = mean_ndcg(df, 'job_id',  col)
    ndcg_cand[name], _ = mean_ndcg(df, 'cand_id', col)

W = 16
names = [n for n,_,_ in MODELS]
print(f'{"":24s}  ' + '  '.join(f'{n:>{W}}' for n in names))
print('─' * (24 + len(names)*(W+2)))
print(f'{"NDCG@5 (by job)":24s}  ' + '  '.join(f'{ndcg_job[n]:{W}.4f}'  for n in names))
print(f'{"NDCG@5 (by candidate)":24s}  ' + '  '.join(f'{ndcg_cand[n]:{W}.4f}' for n in names))

# Per-job breakdown — store for visualisation
job_ndcg_rows = {}
for _, g in df.groupby('job_id'):
    job = g['job_position_name'].iloc[0]
    if len(g) < 2: continue
    true = g['matched_score'].values.reshape(1,-1)
    row = {}
    for name, _, col in MODELS:
        row[name] = ndcg_score(true, g[col].values.reshape(1,-1), k=5)
    job_ndcg_rows[job] = row

print(f'\n{"Job":55s}  {"M0":>7}  {"MA":>7}  {"MB":>7}')
print('─' * 78)
for job, row in sorted(job_ndcg_rows.items(), key=lambda x: -x[1]['Model B']):
    print(f'{job[:55]:55s}  {row["Model 0"]:7.4f}  {row["Model A"]:7.4f}  {row["Model B"]:7.4f}')

                                   Model 0           Model A           Model B
──────────────────────────────────────────────────────────────────────────────
NDCG@5 (by job)                     0.8555            0.8746            0.8864
NDCG@5 (by candidate)               0.9579            0.9641            0.9566

Job                                                           M0       MA       MB
──────────────────────────────────────────────────────────────────────────────
Senior Software Engineer                                  0.8368   0.8156   0.9361
Executive/ Sr. Executive -IT                              0.8447   0.8505   0.9164
Head of Internal Control & Compliance (ICC) - SEVP/DMD    0.8989   0.9421   0.8909
Manager- Human Resource Management (HRM)                  0.8773   0.8998   0.8805
Database Administrator (DBA)                              0.8349   0.8687   0.8740
Asst. Manager/ Manger (Administrative)                    0.8404   0.8711   0.8208


## Visualisations

In [4]:
# Four plots saved to outputs/: score dist, model comparison, NDCG per job, pred vs true
os.makedirs('../outputs', exist_ok=True)
COLORS = {'Model 0':'#4a9aba', 'Model A':'#f5a623', 'Model B':'#e06c5a'}

# ── Plot 1: score distribution ────────────────────────────────────────────────
scores = df['matched_score']
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=40, color='#4a9aba', edgecolor='white', linewidth=0.4)
ax.axvline(scores.mean(),   color='#e86c1e', linestyle='--', linewidth=1.5,
           label=f'mean={scores.mean():.3f}')
ax.axvline(scores.median(), color='#f5a623', linestyle='--', linewidth=1.5,
           label=f'median={scores.median():.3f}')
ax.set_xlabel('matched_score')
ax.set_ylabel('Count')
ax.set_title('Score distribution — matched_score')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/viz_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: viz_score_distribution.png')

# ── Plot 2: grouped bar chart — MAE / RMSE / Spearman ────────────────────────
metric_list = ['MAE', 'RMSE', 'Spearman r']
model_list  = [n for n,_,_ in MODELS]
x     = np.arange(len(metric_list))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(model_list):
    vals = [metrics[name][m] for m in metric_list]
    ax.bar(x + (i-1)*width, vals, width, label=name,
           color=COLORS[name], alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels(metric_list, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Model comparison — pointwise metrics')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/viz_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: viz_model_comparison.png')

# ── Plot 3: NDCG@5 per job — Model A vs B ────────────────────────────────────
sorted_jobs = sorted(job_ndcg_rows.items(), key=lambda x: x[1]['Model B'])
job_names = [j[:45] for j, _ in sorted_jobs]
ndcg_a = [r['Model A'] for _, r in sorted_jobs]
ndcg_b = [r['Model B'] for _, r in sorted_jobs]
y = np.arange(len(job_names))
h = 0.35

fig, ax = plt.subplots(figsize=(10, max(5, len(job_names)*0.4)))
ax.barh(y + h/2, ndcg_a, h, label='Model A', color=COLORS['Model A'], alpha=0.88)
ax.barh(y - h/2, ndcg_b, h, label='Model B', color=COLORS['Model B'], alpha=0.88)
ax.set_yticks(y)
ax.set_yticklabels(job_names, fontsize=7)
ax.set_xlabel('NDCG@5')
ax.set_title('NDCG@5 per job — Model A vs Model B')
ax.axvline(0.5, color='grey', linestyle=':', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/viz_ndcg_per_job.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: viz_ndcg_per_job.png')

# ── Plot 4: predicted vs true — Model A and B ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, _, col) in zip(axes, [m for m in MODELS if m[0] != 'Model 0']):
    yp = df[col].values
    ax.scatter(y_true, yp, alpha=0.05, s=5, color=COLORS[name], rasterized=True)
    lo, hi = min(y_true.min(), yp.min()), max(y_true.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='perfect')
    ax.set_xlabel('True score')
    ax.set_ylabel('Predicted score')
    ax.set_title(f'{name} — {metrics[name]["label"]}')
    ax.legend(fontsize=8)
plt.suptitle('Predicted vs true matched_score', fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/viz_pred_vs_true.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: viz_pred_vs_true.png')

Saved: viz_score_distribution.png


Saved: viz_model_comparison.png
Saved: viz_ndcg_per_job.png


Saved: viz_pred_vs_true.png


## Ablation: job_id one-hot contribution

Retrains each model without the 28 job_id dummy columns to isolate how much of the
performance comes from per-job calibration vs the text and structured features themselves.

In [5]:
# Load ablation results saved by model notebooks (run those first)
import json as _json

def _load_ablation(path):
    try:
        with open(path) as f:
            return _json.load(f)
    except FileNotFoundError:
        return None

aa = _load_ablation('../outputs/ablation_a.json')
ab = _load_ablation('../outputs/ablation_b.json')

def _f(v): return f'{v:.4f}' if v is not None else '    —   '
def _d(a, b): return f'{b-a:+.4f}'.rjust(14) if a is not None and b is not None else f'{"—":>14}'

W = 14
print(f'{"":22s}  {"with job_id":>{W}}  {"without":>{W}}  {"delta":>{W}}')
print('─' * (22 + 3*(W+2)))

rows = [
    ('Model A  Spearman', aa['r_with']    if aa else None, aa['r_without']    if aa else None),
    ('Model B  Spearman', ab['r_with']    if ab else None, ab['r_without']    if ab else None),
    ('Model A  NDCG@5',   aa['ndcg_with'] if aa else None, aa['ndcg_without'] if aa else None),
    ('Model B  NDCG@5',   ab['ndcg_with'] if ab else None, ab['ndcg_without'] if ab else None),
]
for label, with_, without_ in rows:
    print(f'{label:22s}  {_f(with_):>{W}}  {_f(without_):>{W}}  {_d(with_, without_)}')

if not aa or not ab:
    print('\nRun 03_model_a.ipynb and 03_model_b.ipynb ablation cells to populate this table.')

                           with job_id         without           delta
──────────────────────────────────────────────────────────────────────
Model A  Spearman               0.4859          0.5097         +0.0238
Model B  Spearman               0.4504          0.4334         -0.0170
Model A  NDCG@5                 0.8877          0.8798         -0.0079
Model B  NDCG@5                 0.8686          0.8661         -0.0025


Cross-encoder wins NDCG@5 (0.941) and Spearman (0.660) — cross-attention between candidate and job tokens captures matches that separate encodings miss. B2 JobBERT has best MAE (0.109) — the HGB regressor is better calibrated. Model A (Spearman 0.645) is still competitive. Semantic skill imputation helped the cross-encoder (whose primary signal is text) but slightly hurt structured feature models because imputed skills add noise to skill_coverage/fuzzy_skill_coverage.